# theta_interpretation: multi-seed analysis

Loads several seeds of the two `theta_interpretation` runs (`terms=['x2']`
correct vs. `terms=['x2', 'x4']` misspecified), then:

1. Plots `theta_t` and `Theta_reg` over time for one example seed, and the
   mean +/- std across seeds.
2. Computes the normalization constant `Z` three ways and compares them:
   the MGD lower bound (`codes/utils_entropy.py`'s `log_Z_bound`), a direct
   1D-quadrature reference using the actual fitted theta, and the analytic
   Gaussian `Z` (known exactly here, since the data-generating process
   itself is `N(0, data_sigma^2)`).
3. Uses the fitted models to estimate the probability of a rare event
   (`|X| > 4*sigma`) and a typical one (`|X| < 1*sigma`), against the exact
   analytic answer -- this is the sharpest test of whether the misspecified
   `x^4` term actually hurts anything beyond the bulk moments it was never
   needed for.

This notebook only *defines and runs* the analysis -- it generates its own
data and calls the real SDE solver via `run_SDE.py`'s `run_and_diagnose()`,
so running it end-to-end does real (if small, CPU-cheap) MGD fits; nothing
here is precomputed or faked.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy.integrate import quad
from scipy.stats import norm

root = Path.cwd()
if not (root / 'run_SDE.py').exists():
    root = root / 'theta_interpretation'  # allow running the notebook from the repo root
sys.path.insert(0, str(root))
sys.path.insert(0, str(root.parent / 'codes'))

from run_SDE import make_args, run_and_diagnose, get_scalar_potentials, device
from utils_entropy import entropy_bound, log_Z_bound  # codes/utils_entropy.py, on sys.path via codes/

# run_SDE.py sets matplotlib.use('Agg') at import time (needed for its own
# headless script use) -- that silently kills inline display here, so switch
# back after importing from it.
%matplotlib inline

print('device:', device)


## 1. Config and multi-seed runs

`N_SEEDS` independent seeds per potential set. Everything else (data_sigma,
SDE hyperparameters) matches `theta_interpretation.ipynb`'s defaults via
`make_args`'s own defaults -- override here if you want a different sweep.


In [ ]:
N_SEEDS = 10
DATA_SIGMA = 1.0
TERM_SETS = {
    'correct (x2)': ['x2'],
    'misspecified (x2, x4)': ['x2', 'x4'],
}

runs = {label: {} for label in TERM_SETS}
for label, terms in TERM_SETS.items():
    for seed in range(N_SEEDS):
        args = make_args(terms, data_sigma=DATA_SIGMA, seed=seed, outdir=str(root))
        runs[label][seed] = run_and_diagnose(args, outdir=root)
    print(f"{label}: {N_SEEDS} seeds done")


## 2. Theta trajectories: one example seed, then mean +/- std over seeds

`theta_t` and `Theta_reg` both have the same length across seeds for a
given potential set (the time grid and block-averaging schedule are
deterministic in `nt`/`schedule_exponent`/`n_subsample`, not in the random
data draw), so they can be stacked and averaged directly across the seed
axis.


In [ ]:
def stack_over_seeds(runs_for_label, key):
    arrs = [runs_for_label[s]['result'][key].detach().cpu().numpy()
            for s in sorted(runs_for_label)]
    return np.stack(arrs, axis=0)  # (n_seeds, n_steps, n_coef)


def plot_theta_example_and_average(runs_for_label, label, term_names, key, example_seed=0):
    stacked = stack_over_seeds(runs_for_label, key)   # (n_seeds, n_steps, n_coef)
    example = runs_for_label[example_seed]['result'][key].detach().cpu().numpy()
    mean = stacked.mean(0)
    std = stacked.std(0, ddof=1) if stacked.shape[0] > 1 else np.zeros_like(mean)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    steps = np.arange(example.shape[0])
    for i, name in enumerate(term_names):
        axes[0].plot(steps, example[:, i], label=name)
        axes[1].plot(steps, mean[:, i], label=name)
        axes[1].fill_between(steps, mean[:, i] - std[:, i], mean[:, i] + std[:, i], alpha=0.2)
    axes[0].set_title(f'{key}, seed={example_seed} ({label})')
    axes[1].set_title(f'{key}, mean +/- std over {stacked.shape[0]} seeds ({label})')
    for ax in axes:
        ax.set_xlabel('step')
        ax.axhline(0, color='grey', lw=0.5)
        ax.legend()
    fig.tight_layout()
    plt.show()


In [ ]:
for label, terms in TERM_SETS.items():
    term_names = list(get_scalar_potentials(terms).keys())
    plot_theta_example_and_average(runs[label], label, term_names, 'theta_t')
    plot_theta_example_and_average(runs[label], label, term_names, 'Theta_reg')


## 3. Normalization constant: three ways, compared

- **Analytic** (`log_Z_analytic`): the ground-truth value is known exactly
  here, since the data really is `N(0, data_sigma^2)` -- `log Z = 0.5 *
  log(2*pi*data_sigma^2)`, the same number regardless of which potential
  set was used to fit (a correctly-fit misspecified model should recover
  `theta_x4 ~ 0` and hence the same `Z`).
- **Quadrature** (`log_Z_quad`): direct 1D numerical integration of
  `exp(sum_i theta_i * phi_i(x))` using the actual *fitted* theta (not the
  ground truth) -- this is the exact normalizer of whatever density the fit
  actually produced, misspecification and all. Uses the same
  `p(x) ~ exp(+theta^T phi(x))` sign convention as
  `codes/utils_entropy.py` (see its module docstring for the audit).
- **MGD bound** (`log_Z_bound`): the lower bound from `codes/utils_entropy.py`,
  reused as-is -- not reimplemented here.


In [ ]:
def _degree(term):
    return int(term[1:])  # 'x2' -> 2, 'x4' -> 4


def log_p_unnorm(x, theta, term_names):
    return sum(theta[i] * x ** _degree(t) for i, t in enumerate(term_names))


def log_Z_quad(theta, term_names, x_range=(-50.0, 50.0), label=''):
    """log Z via scipy.integrate.quad, log-space-stabilized against a
    coarse-grid max estimate (same idea as the bimodal validation in
    codes/utils_entropy.py's module docstring, generalized to any subset of
    monomials).

    At this experiment's toy scale, a misspecified fit's theta_x4 can come
    out POSITIVE by chance (undertrained -- the quartic coefficient isn't
    pinned down by data that doesn't need it), which makes
    exp(theta^T phi(x)) diverge as |x| -> infinity instead of decay: not
    normalizable, log Z is +infinity. Detected here as the coarse grid's
    log-density still increasing at the boundary (argmax at either end of
    the grid) -- returns nan with a printed warning rather than silently
    propagating -inf/nan into every downstream average."""
    grid = np.linspace(*x_range, 4001)
    log_vals = log_p_unnorm(grid, theta, term_names)
    argmax = int(log_vals.argmax())
    if argmax in (0, len(grid) - 1):
        print(f"  WARNING log_Z_quad{' ' + label if label else ''}: unnormalizable density "
              f"for theta={theta} (log-density still rising at the grid boundary, "
              f"x={grid[argmax]:.1f} -- likely a non-confining, e.g. positive, "
              f"leading-order coefficient). Returning nan.")
        return float('nan')

    m = log_vals[argmax]

    def integrand(x):
        return np.exp(log_p_unnorm(x, theta, term_names) - m)

    val, _ = quad(integrand, *x_range, limit=200)
    if val <= 0 or not np.isfinite(val):
        print(f"  WARNING log_Z_quad{' ' + label if label else ''}: quad integral "
              f"non-positive/non-finite (val={val}) for theta={theta}. Returning nan.")
        return float('nan')
    return m + np.log(val)


def tail_probability(theta, term_names, log_Z, a, b=np.inf):
    """P(a < X < b) under the FITTED (properly normalized, via log_Z)
    density. nan in, nan out (no quad call) if log_Z itself is nan -- see
    log_Z_quad."""
    if not np.isfinite(log_Z):
        return float('nan')

    def integrand(x):
        return np.exp(log_p_unnorm(x, theta, term_names) - log_Z)
    val, _ = quad(integrand, a, b, limit=200)
    return val


In [ ]:
log_Z_analytic = 0.5 * np.log(2 * np.pi * DATA_SIGMA ** 2)
print(f"log_Z_analytic = {log_Z_analytic:.6f}\n")

Z_rows = []
for label, terms in TERM_SETS.items():
    term_names = list(get_scalar_potentials(terms).keys())
    potentials = get_scalar_potentials(terms)

    quad_vals, bound_vals = [], []
    for seed, run in runs[label].items():
        theta_1 = run['result']['Theta_reg'][-1].detach().cpu().numpy()
        quad_vals.append(log_Z_quad(theta_1, term_names, label=f'({label}, seed={seed})'))

        single = {f'seed{seed}': run['result']}
        out = log_Z_bound(single, f'seed{seed}', 'Theta_reg', potentials, device=device)
        bound_vals.append(out['log_Z_bound'])

    quad_vals = np.array(quad_vals)
    bound_vals = np.array(bound_vals)
    n_valid = int(np.isfinite(quad_vals).sum())
    print(f"--- {label} (terms={term_names}) ---")
    if n_valid < len(quad_vals):
        print(f"  {len(quad_vals) - n_valid}/{len(quad_vals)} seeds gave an unnormalizable "
              f"fit (see WARNING above) -- excluded from log_Z_quad's mean/std below via nanmean/nanstd.")
    print(f"  log_Z_quad  (fitted theta, mean +/- std over {n_valid} valid seeds): "
          f"{np.nanmean(quad_vals):.4f} +/- {np.nanstd(quad_vals, ddof=1):.4f}")
    print(f"  log_Z_bound (MGD lower bound, mean +/- std over seeds): "
          f"{bound_vals.mean():.4f} +/- {bound_vals.std(ddof=1):.4f}")
    print(f"  log_Z_analytic (ground truth):                          {log_Z_analytic:.4f}")
    print(f"  gap (analytic - bound), should be >= 0 on average:      "
          f"{log_Z_analytic - bound_vals.mean():.4f}")
    print()
    Z_rows.append({'label': label, 'log_Z_quad_mean': quad_vals.mean(), 'log_Z_quad_std': quad_vals.std(ddof=1),
                    'log_Z_bound_mean': bound_vals.mean(), 'log_Z_bound_std': bound_vals.std(ddof=1)})


## 4. Rare-event vs. typical-event probability

Rare event: `|X| > 4*data_sigma`. Typical event: `|X| < 1*data_sigma`
(these thresholds are just defaults -- change `RARE_THRESHOLD`/
`TYPICAL_THRESHOLD` below).

Two different "probability under the fit" numbers are reported, and they
are NOT the same thing:

- **fitted-model probability** (`P_quad`): uses `log_Z_quad`, the exact
  normalizer of the density the fit actually produced. This tells you how
  good the FIT is (bulk + tails), independent of the MGD bound's own
  looseness.
- **bound-implied probability** (`P_bound_upper`): plugs the MGD
  `log_Z_bound` in for `Z` instead. Since `log_Z_bound <= log_Z_true`,
  using it *overstates* Z's reciprocal, i.e. `1/Z_bound >= 1/Z_true`, so
  this number is an UPPER BOUND on the true probability, not an estimate --
  useful to see how much a naive use of the bound would distort a
  rare-event estimate, not a candidate for "the" probability.


In [ ]:
RARE_THRESHOLD = 4 * DATA_SIGMA
TYPICAL_THRESHOLD = 1 * DATA_SIGMA

P_rare_true = 2 * norm.sf(RARE_THRESHOLD / DATA_SIGMA)         # two-sided |X| > 4 sigma
P_typical_true = 2 * norm.cdf(TYPICAL_THRESHOLD / DATA_SIGMA) - 1  # |X| < 1 sigma

print(f"True (analytic) P(|X| > {RARE_THRESHOLD}) = {P_rare_true:.6e}")
print(f"True (analytic) P(|X| < {TYPICAL_THRESHOLD}) = {P_typical_true:.6e}\n")

for label, terms in TERM_SETS.items():
    term_names = list(get_scalar_potentials(terms).keys())
    potentials = get_scalar_potentials(terms)

    P_rare_quad, P_typ_quad, P_rare_bound = [], [], []
    for seed, run in runs[label].items():
        theta_1 = run['result']['Theta_reg'][-1].detach().cpu().numpy()
        lZ_quad = log_Z_quad(theta_1, term_names)  # WARNING already printed once per seed in section 3 above

        if not np.isfinite(lZ_quad):
            # theta_1 itself is non-confining (e.g. positive leading coefficient):
            # exp(theta_1^T phi(x)) diverges as |x|->infinity, so NEITHER
            # Z (quad or MGD bound) gives a well-defined tail probability --
            # skip rather than let quad return a raw inf below.
            P_rare_quad.append(np.nan)
            P_typ_quad.append(np.nan)
            P_rare_bound.append(np.nan)
            continue

        p_rare = (tail_probability(theta_1, term_names, lZ_quad, RARE_THRESHOLD, np.inf)
                  + tail_probability(theta_1, term_names, lZ_quad, -np.inf, -RARE_THRESHOLD))
        p_typ = tail_probability(theta_1, term_names, lZ_quad, -TYPICAL_THRESHOLD, TYPICAL_THRESHOLD)
        P_rare_quad.append(p_rare)
        P_typ_quad.append(p_typ)

        single = {f'seed{seed}': run['result']}
        lZ_bound = log_Z_bound(single, f'seed{seed}', 'Theta_reg', potentials, device=device)['log_Z_bound']
        p_rare_bound = (tail_probability(theta_1, term_names, lZ_bound, RARE_THRESHOLD, np.inf)
                         + tail_probability(theta_1, term_names, lZ_bound, -np.inf, -RARE_THRESHOLD))
        P_rare_bound.append(p_rare_bound)

    P_rare_quad = np.array(P_rare_quad)
    P_typ_quad = np.array(P_typ_quad)
    P_rare_bound = np.array(P_rare_bound)
    n_valid = int(np.isfinite(P_rare_quad).sum())

    print(f"--- {label} ({n_valid}/{len(P_rare_quad)} seeds with a normalizable fit) ---")
    print(f"  P_quad(|X|>{RARE_THRESHOLD})  = {np.nanmean(P_rare_quad):.6e} +/- {np.nanstd(P_rare_quad, ddof=1):.2e}  "
          f"(true: {P_rare_true:.6e})")
    print(f"  P_quad(|X|<{TYPICAL_THRESHOLD})  = {np.nanmean(P_typ_quad):.6e} +/- {np.nanstd(P_typ_quad, ddof=1):.2e}  "
          f"(true: {P_typical_true:.6e})")
    print(f"  P_bound_upper(|X|>{RARE_THRESHOLD}) = {np.nanmean(P_rare_bound):.6e} +/- {np.nanstd(P_rare_bound, ddof=1):.2e}  "
          f"(NOT an estimate -- upper bound via log_Z_bound, see markdown above)")
    print()


## Summary

If the fit is well-behaved: both potential sets' `log_Z_quad` should land
close to `log_Z_analytic`, and both should give rare/typical-event
probabilities close to the true Gaussian ones -- meaning the spurious `x^4`
term in the misspecified model, despite changing `theta`'s values and the
normalization constant along the way, washes out once you actually
normalize and integrate. If it does NOT wash out (misspecified model's
`P_quad` visibly off from the correct model's), that is the concrete,
quantitative cost of the misspecification this whole experiment was set up
to probe.
